In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score, cross_validate
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, f1_score, ConfusionMatrixDisplay
from itertools import combinations
from tqdm.notebook import tqdm
from pathlib import Path
import warnings

# model

In [ ]:
def combine_categories(df: pd.DataFrame, combination_map: dict, target_col: str = 'FL_UDSD') -> pd.DataFrame:
    """
    Combine multiple categories in the target column into single categories.
    
    Args:
        df: Input dataframe
        target_col: Column containing categories to combine
        combination_map: Dictionary where keys are new category names and values are lists of 
                        categories to combine into that new category.
                        Example: {'SCD/Impaired': ['Subjective Cognitive Decline', 'Impaired Not SCD/MCI'],
                                 'Normal/SCD': ['Normal cognition', 'Subjective Cognitive Decline']}
    
    Returns:
        pd.DataFrame: DataFrame with combined categories
    """
    df = df.copy()
    
    # Apply each combination
    for new_category, old_categories in combination_map.items():
        df[target_col] = df[target_col].replace(old_categories, new_category)
    
    return df

In [ ]:
df = pd.read_csv('../data/clinical/clinical_preprocessed.csv')
df.info()

In [ ]:
diagnosis_order = ['Normal cognition', 'Subjective Cognitive Decline', 'Impaired Not SCD/MCI',
                   'Early MCI', 'Late MCI', 'Dementia']
diagnosis_map = {i+1: label for i, label in enumerate(diagnosis_order)}

diagnosis_map

In [ ]:
# df['FL_UDSD_CAT'] = df['FL_UDSD'].map(diagnosis_map)
# df['FL_UDSD_CAT']

## Removing some columns
These columns are for informative purposes ['CDRGLOB','AMYLPET', 'NACCETPR'],

In [ ]:
REMOVE_FEATURES = ['CDRGLOB','AMYLPET', 'NACCETPR']
df_filter = df.drop(columns=REMOVE_FEATURES).dropna()
df_filter.info()

In [ ]:
def exhaustive_feature_search(
    X, y, groups, cv, estimator,
    min_features=2, max_features=None,
    scoring=('f1_macro', 'balanced_accuracy'),
    primary_metric='f1_macro',
    n_jobs=-1,
):
    """Evaluate every feature combination with size in [min_features, max_features].

    Iterates over all subsets of the columns of X whose cardinality falls
    within [min_features, max_features] and scores each one with grouped
    cross-validation. Useful for small feature sets where brute-force search
    is tractable; cost grows as the sum of binomial coefficients over the
    requested range, so mind the combinatorial explosion.

    Note on scores: uses sklearn.model_selection.cross_validate, which
    reports held-out TEST scores per fold (not training scores). Each
    combination is scored on every fold's test split and aggregated to
    mean/std, and per-fold scores are also returned.

    Parameters
    ----------
    X : pandas.DataFrame
        Feature matrix. Column names are used to enumerate subsets.
    y : array-like of shape (n_samples,)
        Target vector aligned with X.
    groups : array-like of shape (n_samples,)
        Group labels passed to cv so samples from the same group stay on
        the same side of each split (e.g. patient IDs to prevent leakage).
    cv : cross-validation splitter
        Any splitter accepting groups (e.g. StratifiedGroupKFold).
    estimator : sklearn ensemble model
    min_features : int, default=2
        Smallest subset size to evaluate (inclusive).
    max_features : int or None, default=None
        Largest subset size to evaluate (inclusive). None means all
        columns of X.
    scoring : tuple/list/dict of str, default=('f1_macro', 'balanced_accuracy')
        Any scoring strings accepted by sklearn.model_selection.cross_validate.
    primary_metric : str, default='f1_macro'
        Metric used to sort the results. Must appear in `scoring`.
    n_jobs : int, default=-1
        Parallelism passed to cross_validate. -1 uses all cores.

    Returns
    -------
    pandas.DataFrame
        One row per subset, sorted by mean_{primary_metric} descending.
        Columns: n_features, features (list of column names), and for each
        metric m in `scoring`:
          - mean_{m}, std_{m}
          - fold{i}_{m} for i in 0..n_splits-1 (per-fold test score)
    """
    all_features = X.columns.tolist()
    if max_features is None:
        max_features = len(all_features)

    metrics = list(scoring) if not isinstance(scoring, dict) else list(scoring.keys())
    if primary_metric not in metrics:
        raise ValueError(f"primary_metric={primary_metric!r} not in scoring={metrics}")

    results = []
    for k in range(min_features, max_features + 1):
        combos = list(combinations(all_features, k))
        print(f"Evaluating {len(combos)} combinations of size {k}...")
        for feats in tqdm(combos, leave=False):
            feats = list(feats)
            cv_res = cross_validate(
                estimator, X[feats], y,
                groups=groups, cv=cv, scoring=scoring, n_jobs=n_jobs,
                return_train_score=False,
            )
            row = {'n_features': k, 'features': feats}
            for m in metrics:
                fold_scores = cv_res[f'test_{m}']
                row[f'mean_{m}'] = fold_scores.mean()
                row[f'std_{m}']  = fold_scores.std()
                for i, s in enumerate(fold_scores):
                    row[f'fold{i}_{m}'] = s
            results.append(row)

    return (
        pd.DataFrame(results)
          .sort_values(f'mean_{primary_metric}', ascending=False)
          .reset_index(drop=True)
    )


In [ ]:
X = df_filter.drop(columns=['PTID', "VISITYR", 'FL_UDSD'])
y = (df_filter["FL_UDSD"].astype(int) - 1)  # xgboost wants 0-indexed int labels

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
model = xgb.XGBClassifier(
    n_estimators=100,
    random_state=42,
    objective='multi:softprob',
    eval_metric='mlogloss',
    tree_method='hist',
    n_jobs=4,
)

In [ ]:
X.info()

In [ ]:
y.value_counts()

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
    combo_results = exhaustive_feature_search(
        X, y,
        groups=df_filter['PTID'],
        cv=cv,
        estimator=model,
        min_features=2,
        max_features=len(X.columns),
    )

In [ ]:
combo_results.to_csv("results/xgb_results.csv", index=False)

In [ ]:
SCD_CODE = 2
IMPAIRED_NOT_SCD_CODE = 3
PRE_MCI_CODE = 2

df_combined = df_filter.copy()

df_combined = combine_categories(
    df_combined,
    combination_map={PRE_MCI_CODE: [SCD_CODE, IMPAIRED_NOT_SCD_CODE]},
    target_col='FL_UDSD',
)

# Renumber so codes are contiguous: 1=Normal, 2=Pre-MCI, 3=EMCI, 4=LMCI, 5=Dementia
renumber_map = {1: 1, 2: 2, 4: 3, 5: 4, 6: 5}
df_combined['FL_UDSD'] = df_combined['FL_UDSD'].map(renumber_map)

df_combined['FL_UDSD'].value_counts().sort_index()

In [ ]:
diagnosis_order = ['Normal cognition', 'Pre-MCI', 
                   'Early MCI', 'Late MCI', 'Dementia']
diagnosis_map = {i+1: label for i, label in enumerate(diagnosis_order)}

diagnosis_map

In [ ]:
X = df_filter.drop(columns=['PTID', "VISITYR", 'FL_UDSD'])
y = (df_filter["FL_UDSD"].astype(int) - 1)  # xgboost wants 0-indexed int labels

In [ ]:
X_combined = df_combined.drop(columns=['PTID', 'VISITYR', 'FL_UDSD'])
y_combined = (df_combined['FL_UDSD'].astype(int) - 1)

cv_combined = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
model_combined = xgb.XGBClassifier(
    n_estimators=100,
    random_state=42,
    objective='multi:softprob',
    eval_metric='mlogloss',
    tree_method='hist',
    n_jobs=4,
)

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
    combo_results_combined = exhaustive_feature_search(
        X_combined, y_combined,
        groups=df_combined['PTID'],
        cv=cv_combined,
        estimator=model_combined,
        min_features=2,
        max_features=len(X_combined.columns),
    )

In [ ]:
combo_results.to_csv("results/xgb_results_SCD_Imp.csv", index=False)